# Multimodal

**Tags:** architecture, vision
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/multimodal.md

## TL;DR

LLMs that understand multiple input modalities: text, images, audio, video. Encode each modality separately, project to shared embedding space, concatenate sequences. Feed to language model. Enables vision+language reasoning. Key challenge: token efficiency (images = 500+ tokens) and modality alignment.

## Core Intuition

Humans understand images and text together. Multimodal LLMs bridge this: convert images to embeddings (vectors), concatenate with text tokens, feed to LLM. Key insight: vision encoders (ViT, CLIP) compress images to ~100-500 embedding tokens—same token format as text. This lets standard LLMs process both.

## How It Works

**Architecture Overview:**

```
Input: Image + Prompt
  ↓
Image Encoding:
  Image (3×H×W) → Vision Encoder → patch embeddings
  → compress to 100-500 tokens
  ↓
Text Encoding:
  Prompt text → tokenizer → token IDs
  ↓
Projection:
  image_tokens (768-dim) → text_embed_space (4096-dim)
  ↓
Concatenate:
  [image_tokens (projected), text_tokens]
  → sequence of length 100-500 + prompt_length
  ↓
LLM Processing:
  Pass through transformer, generate response
```

**Vision Encoders (by architecture):**

**1. Vision Transformer (ViT):**
```
Image → patch embeddings (16×16 patches)
  → transformer encoder (12-24 layers)
  → [CLS] token (image representation)

Example (CLIP ViT-L):
  Input: 224×224 RGB image
  Patches: 16×16 = 196 patches (7×7 grid)
  Output: 768-dim vector
  
Pros: state-of-art, parallelizable
Cons: needs significant compute, large models
```

**2. CNN-based (ResNet):**
```
Image → convolutional layers → feature maps
  → global average pooling → vector

Example (ResNet-50):
  Input: 224×224
  Output: 2048-dim vector
  
Pros: smaller, faster
Cons: less expressive, local receptive field
```

**3. Dense Visual Features (for tokens):**
```
Instead of single vector per image:
  → preserve spatial information (grid of vectors)
  → pass ~100-500 tokens to LLM

Example (CLIP + pooling):
  ViT output: 196 patch embeddings (16×16)
  Adaptive pooling: 100-144 tokens
  
Pros: more information, better for details
Cons: higher token count, slower LLM inference
```

**Projection Layer:**
```
Vision encoder output: 768-dim
LLM embedding space: 4096-dim

Linear projection: (768,) → (4096,)
Single matrix multiply

Parameters: 768 × 4096 = ~3.1M (small, trainable)

Optional: multi-layer MLP for more expressiveness
  Vision output → FC layer (768→2048) → ReLU
           → FC layer (2048→4096) → LLM space
```

**Integration Methods:**

**1. Early Fusion (concat at input):**
```
[visual_tokens, text_tokens] → LLM
+ allows cross-modal attention
- requires retraining LLM on multimodal data
```

**2. Late Fusion (parallel encoders):**
```
Image → Vision Encoder → features
Text → LLM → features
Combine features at output layer
+ vision encoder frozen, LLM frozen
- less cross-modal interaction, typically weaker
```

**3. Mid-layer Fusion (adapter):**
```
Image → Vision Encoder → adapter → text_space
Text → Text Encoder
Concatenate before final LLM layers
+ balance between parameters and expressiveness
- requires some LLM tuning
```

**Examples of Popular Architectures:**

```
GPT-4V / Claude Vision:
  Vision: Custom ViT-based encoder
  LLM: Large causal transformer
  Method: Early fusion
  Token cost: ~85 tokens per image
  
CLIP:
  Vision: ViT (frozen)
  Text: Text encoder (frozen)
  Method: Contrastive learning (not generative)
  Use case: retrieval, classification
  
LLaVA (Large Language and Vision Assistant):
  Vision: CLIP ViT-L encoder
  Projection: Linear layer (768→4096)
  LLM: Llama-2 (frozen or fine-tuned)
  Total params: 13B (7B Llama + 5M projection + CLIP vision)
  
PaliGemma:
  Vision: Pre-trained ViT
  LLM: Gemma (small, 3B)
  Method: Early fusion with co-training
```

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Multimodal Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | Text-Only | Text+Image | Text+Image+Audio | Full Multimodal |
|--------|-----------|-----------|------------------|-----------------|
| Model size | 7B | 13B-70B | 20B-100B | 50B-300B |
| Training data | 1B tokens | 1B text + 100M images | Above + 500k hours audio | Diverse, large-scale |
| Inference latency | 20ms/token | 30ms/token (+50%) | 40ms/token (+100%) | 50ms/token (+150%) |
| Quality (MMLU) | 85% | 88-92% | 88-93% | 90-95% |
| Context window | 4-200K tokens | Same (images count!) | Same | Same |
| Cost | Cheap | 2-3x (more tokens) | 3-5x | 5-10x |

**Image Token Costs:**
```
Architecture | Tokens per image |  Latency per image
CLIP ViT-L  | 100-144          |  ~10ms
GPT-4V      | 85 (estimated)   |  ~5ms
Dense grid  | 196-576          |  ~15ms (too many)
Compressed  | 32-64 (aggressive) |  ~3ms (quality risk)
```

### Code Implementation

```python
# Key imports for Multimodal
import numpy as np
import torch
from typing import Any

# Multimodal example implementation
class Multimodal:
    """
    Multimodal implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Multimodal"]
    A -->|used with| D["Embeddings"]
    A -->|used with| D["Tokenization"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Multimodal used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Multimodal?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Multimodal compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Multimodal?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Multimodal?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/multimodal.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]